# 25

In [1]:
import pandas as pd

Create a data frame from the file nyc-parking-violations-2020.csv. We are only
interested in a handful of the columns:
- `Plate ID`
- `Registration State`
- `Vehicle Make`
- `Vehicle Color`
- `Violation Time`
- `Street Name`

In [2]:
df = pd.read_csv('./data/nyc-parking-violations-2020.csv', usecols=['Plate ID', 'Registration State', 'Vehicle Make', 'Vehicle Color', 'Violation Time', 'Street Name'])

How many rows are in the data frame when it is read into memory?

In [3]:
df.info(show_counts=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12495734 entries, 0 to 12495733
Data columns (total 6 columns):
 #   Column              Non-Null Count     Dtype 
---  ------              --------------     ----- 
 0   Plate ID            12495532 non-null  object
 1   Registration State  12495734 non-null  object
 2   Vehicle Make        12433314 non-null  object
 3   Violation Time      12495456 non-null  object
 4   Street Name         12494317 non-null  object
 5   Vehicle Color       12103752 non-null  object
dtypes: object(6)
memory usage: 572.0+ MB


Remove rows with any missing data (i.e., a NaN value). How many rows remain after doing this pruning?

In [4]:
df_one = df.dropna()

In [5]:
len(df_one.index)

12048375

 If each parking ticket brings $100 into the city, and missing data means the ticket can be successfully contested, how much money may New York City lose due to such missing data?

In [6]:
f'${(len(df.index) - len(df_one.index)) * 100:,}'

'$44,735,900'

Let’s instead assume that a ticket can only be dismissed if the license plate,
state, car make, and/or street name are missing. Remove rows that are missing
one or more of these. How many rows remain?

In [7]:
df_two = df.dropna(subset=['Plate ID', 'Registration State', 'Vehicle Make', 'Street Name'])
len(df_two.index)

12431949

Assuming $100/ticket, how much money would the city lose as a result of this missing data?

In [8]:
f'${(len(df.index) - len(df_two.index)) * 100:,}'

'$6,378,500'

Now let’s assume that tickets can be dismissed if the license plate, state, and/or
street name are missing. Remove rows that are missing one or more of these.
How many rows remain?

In [9]:
df_three = df.dropna(subset=['Plate ID', 'Registration State', 'Street Name'])
len(df_three.index)

12494116

Assuming $100/ticket, how much money would the city lose as a result of this missing data?

In [10]:
f'${(len(df.index) - len(df_three.index)) * 100:,}'

'$161,800'

How many rows would you eliminate if you required at least three non-null values from the four columns Plate ID, Registration State, Vehicle Make, and Street Name?

In [11]:
df_four = df.dropna(subset=['Plate ID',
                            'Registration State',
                            'Vehicle Make',
                            'Street Name'],
                    thresh=3)
len(df.index) - len(df_four.index)

253

Which of the columns you’ve imported has the greatest number of `NaN` values? Is this a problem?

In [12]:
df.isnull().sum()

Plate ID                 202
Registration State         0
Vehicle Make           62420
Violation Time           278
Street Name             1417
Vehicle Color         391982
dtype: int64

In [13]:
nan_plate = df.replace({'Plate ID':{'BLANKPLATE': None}})

Null data is bad, but there is plenty of bad non-null data, too. For example, many cars with `BLANKPLATE` as a plate ID were ticketed. Turn these into NaN values, and rerun the previous query.

In [14]:
nan_plate.isnull().sum()

Plate ID                9084
Registration State         0
Vehicle Make           62420
Violation Time           278
Street Name             1417
Vehicle Color         391982
dtype: int64

# 26

Create a data frame from the file celebrity_deaths_2016.csv.

Keep only the columns `dateofdeath` and `age`.

In [168]:
df = pd.read_csv('./data/celebrity_deaths_2016.csv', usecols=['dateofdeath', 'age'])

Create a new column called `month` that contains the month of death, and set this as the index. Sort the data frame by this index.

In [169]:
df['month'] = df['dateofdeath'].str.slice(5,7)
df = df.set_index('month')
df = df.sort_index()

Check how many rows are missing data in the `age` column. What percentage of the data is missing?

In [170]:
print(f"{df['age'].isnull().sum() / len(df) * 100:.2f}%")

0.41%


- Drop the rows with missing data in the `age` column
- Convert the `age` column to a numeric type
- Remove any rows with an age greater than 120
- What is the average age at the time of death?

In [171]:
df = df.dropna(subset=['age'])
df['age'] = pd.to_numeric(df['age'], errors='coerce')
df = df.loc[df['age'] < 120]
df.loc['02':'07', 'age'].mean()

np.float64(77.17887409200968)

In [172]:
df['age'].describe()

count    6481.000000
mean       77.019287
std        16.428815
min         7.000000
25%        69.000000
50%        81.000000
75%        89.000000
max       116.000000
Name: age, dtype: float64

Add a new column called `day` that contains the day of death. Set a multi-index with the month and day columns, and sort the data frame by this index. Get the average age of celebrities who died between February 15th and July 15th.

In [173]:
# Get the month, in slice [5:7]
df['month'] = df['dateofdeath'].str.slice(5,7)
# Get the day, in slice [8:]
df['day'] = df['dateofdeath'].str.slice(8,None)
# Set a multi-index
df = df.set_index(['month', 'day'])
# Sort the index
df = df.sort_index()
# Get the rows from Feb 15th through July 15th, and the 'age' column, then the average
df.loc[('02', '15'):('07', '15'), 'age'].mean()

np.float64(77.05183037332367)

Add a new column called `causeofdeath` that contains the cause of death.

In [174]:
df = pd.read_csv('./data/celebrity_deaths_2016.csv', usecols=['dateofdeath', 'age', 'causeofdeath'])

In [176]:
df.info(show_counts=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6543 entries, 0 to 6542
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   dateofdeath   6543 non-null   object
 1   age           6516 non-null   object
 2   causeofdeath  1535 non-null   object
dtypes: object(3)
memory usage: 153.5+ KB


Fill in the missing values in the `causeofdeath` column with the string `unknown`.

What are the 10 most common causes of death?

In [109]:
df['causeofdeath'] = df['causeofdeath'].fillna('unknown')
df['causeofdeath'].value_counts().head(10)

causeofdeath
unknown               5008
 cancer                248
 heart attack          125
 traffic collision      56
 lung cancer            51
 pneumonia              50
 heart failure          49
 shot                   42
 stroke                 36
 pancreatic cancer      35
Name: count, dtype: int64

# 27

Load the `titanic3.xls` file into a data frame. Check for missing data.

In [140]:
df = pd.read_excel('./data/titanic3.xls')

Determine which columns have missing data, and how many rows are missing data in each column.

In [142]:
df.columns[df.isnull().sum() > 0 ]

Index(['age', 'fare', 'cabin', 'embarked', 'boat', 'body', 'home.dest'], dtype='object')

In [143]:
df.isnull().sum()

pclass          0
survived        0
name            0
sex             0
age           263
sibsp           0
parch           0
ticket          0
fare            1
cabin        1014
embarked        2
boat          823
body         1188
home.dest     564
dtype: int64

Fill in the missing values in the `age` column with the mean of the column.

In [144]:
df['age'] = df['age'].fillna(df['age'].mean())

Drop the rows with missing values in the `fare` and `embarked` columns.

In [145]:
df = df.dropna(subset=['fare', 'embarked'])

Fill in the missing values in the `home.dest` column with the most common value for that column.

In [146]:
df['home.dest'] = df['home.dest'].fillna(df['home.dest'].mode()[0])

In [147]:
df['home.dest'].isna().sum()

np.int64(0)

Create a series called `most_common_destinations` in which the index is the unique values in the `embarked` column, and the values are the most common values in the `home.dest` column.

In [148]:
most_common_destinations = pd.Series([], dtype=object)

for embarked_value in df['embarked'].dropna().unique():
    most_common_destinations.loc[embarked_value] = df.loc[df['embarked']==embarked_value, 'home.dest'].value_counts().index[0]

most_common_destinations

S    New York, NY
C    New York, NY
Q    New York, NY
dtype: object

Use `most_common_destinations` to fill in the missing values in the `home.dest` column. If a row has a missing value in `home.dest`, but not in `embarked`, use the value from `most_common_destinations` that corresponds to the value in `embarked`. If both are missing, leave it as is.

In [149]:
df['home.dest'] = df.apply(lambda x: most_common_destinations[x['embarked']] if pd.isnull(x['home.dest']) else x['home.dest'], axis=1)

In [150]:
df['home.dest'].isnull().sum()

np.int64(0)

In [151]:
df

,pclass,survived,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked,boat,body,home.dest
0,1,1,"Allen, Miss. Elisabeth Walton",female,29.000000,0,0,24160,211.3375,B5,S,2,NaN,"St Louis, MO"
1,1,1,"Allison, Master. Hudson Trevor",male,0.916700,1,2,113781,151.5500,C22 C26,S,11,NaN,"Montreal, PQ / Chesterville, ON"
2,1,0,"Allison, Miss. Helen Loraine",female,2.000000,1,2,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"
3,1,0,"Allison, Mr. Hudson Joshua Creighton",male,30.000000,1,2,113781,151.5500,C22 C26,S,NaN,135.0,"Montreal, PQ / Chesterville, ON"
4,1,0,"Allison, Mrs. Hudson J C (Bessie Waldo Daniels)",female,25.000000,1,2,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1304,3,0,"Zabour, Miss. Hileni",female,14.500000,1,0,2665,14.4542,NaN,C,NaN,328.0,"New York, NY"
1305,3,0,"Zabour, Miss. Thamine",female,29.881135,1,0,2665,14.4542,NaN,C,NaN,NaN,"New York, NY"
1306,3,0,"Zakarian, Mr. Mapriededer",male,26.500000,0,0,2656,7.2250,NaN,C,NaN,304.0,"New York, NY"
1307,3,0,"Zakarian, Mr. Ortin",male,27.000000,0,0,2670,7.2250,NaN,C,NaN,NaN,"New York, NY"


# 28

Create a data frame from the file `nyc-parking-violations-2020.csv`. Load the following columns:
- `Plate ID`
- `Registration State`
- `Vehicle Make`
- `Vehicle Color`
- `Street Name`

In [177]:
filename = './data/nyc-parking-violations-2020.csv'

df = pd.read_csv(filename,
                 usecols=['Plate ID',  'Registration State',
                        'Vehicle Make', 'Vehicle Color', 'Street Name'])

df.head()

,Plate ID,Registration State,Vehicle Make,Street Name,Vehicle Color
0,J58JKX,NJ,HONDA,43 ST,BK
1,KRE6058,PA,ME/BE,UNION ST,BLK
2,444326R,NJ,LEXUS,CLERMONT AVENUE,BLACK
3,F728330,OH,CHEVR,DIVISION AVE,NaN
4,FMY9090,NY,JEEP,GRAND ST,GREY


Check a sample of the `Vehicle Color` column. How many unique values are there? What are the most common values?

In [155]:
import numpy as np

In [178]:
np.random.seed(0)
df['Vehicle Color'].sample(30)

2752511       RED
964568       BLUE
5049760        BK
4248515        GY
6899272       RED
11549116       BL
9816025        BK
11346931       WH
2772528        BK
3663790        WH
7179841        WH
661128       BLUE
5401291     BLACK
526821      WHITE
4014922        BK
6166605       RED
11905504    BLACK
428139         BK
5328844     WHITE
492842      SILVE
920426         BL
1979514        GY
2105457        WH
11996578       RD
3992380        WH
12480694       GY
1009122        GY
9539409        WH
11521484       BK
8954222     WHITE
Name: Vehicle Color, dtype: object

In [179]:
len(df['Vehicle Color'].value_counts().index)

1896

In [180]:
df['Vehicle Color'].value_counts().head(30)

Vehicle Color
WH       2344858
GY       2307704
BK       2066374
WHITE    1061234
BL        775124
RD        483298
BLACK     465110
GREY      306787
BROWN     292348
SILVE     191477
GR        182929
BLUE      178298
RED       161693
TN        120576
BR        102204
YW         98700
BLK        91539
OTHER      60245
GREEN      58765
GL         54851
GRY        46527
MR         42812
GRAY       40854
WHT        35433
YELLO      32792
WHI        29760
OR         28100
BK.        27830
WT         25583
WT.        24593
Name: count, dtype: int64

Create a dictionary to clean up the `Vehicle Color` column and then apply it:

In [181]:
colormap = {'WH': 'WHITE',
          'GY':'GRAY',
             'BK':'BLACK',
             'BL':'BLUE',
             'RD':'RED',
             'GR':'GRAY',
             'TN':'TAN',
             'BR':'BROWN',
             'YW':'YELLO',
             'BLK':'BLACK',
             'GRY':'GRAY',
             'WHT':'WHITE',
             'WHI':'WHITE',
             'OR':'ORANG',
             'BK.':'BLACK',
             'WT':'WHITE',
            'WT.':'WHITE'}

In [182]:
df['Vehicle Color'] = df['Vehicle Color'].replace(colormap)

In [183]:
df['Vehicle Color'].value_counts().shape[0]

1879

In [184]:
df['Vehicle Color'].value_counts().head(50)

Vehicle Color
WHITE    3521461
BLACK    2650853
GRAY     2578014
BLUE      953422
RED       644991
BROWN     394552
GREY      306787
SILVE     191477
TAN       141667
YELLO     131492
OTHER      60245
GREEN      58765
GL         54851
MR         42812
ORANG      39606
GY.        22460
GOLD       21687
SIL        20116
BLU        15240
SL.        13145
LTGY       13055
SL         10343
LTG        10093
BL.         9649
LT/         8976
PR          7518
DK/         7498
W           7367
RD.         7128
DKGY        6004
GYGY        5039
BLK.        4853
GRN         4829
B           4145
WH.         3811
BRO         3802
DKG         3702
PURPL       3635
BRN         3582
BKGY        3504
WHBL        3489
DKBL        2912
GN          2883
WHT.        2796
BN          2787
BLUE.       2638
WHGY        2381
UNKNO       2205
RED.        2141
BRW         2081
Name: count, dtype: int64

Write a function that, given a value, cleans up the `Vehicle Make` data: putting the name in all caps, removing punctuation, and standardizing whatever names you can.

Then use the `apply` method to fix the column. How many distinct vehicle makes are there when you’re done?

In [185]:
import string

def clean_name(one_string):

    if not isinstance(one_string, str):
        return one_string

    output = ''

    for one_character in one_string.strip().upper():
        if one_character in string.ascii_uppercase:
            output += one_character

    return output

print(len(df['Vehicle Make'].value_counts()))
df['Vehicle Make'] = df['Vehicle Make'].apply(clean_name)
print(len(df['Vehicle Make'].value_counts()))

5210
4915


How standardized are the street names in the data set? What changes could you apply to improve things?

In [186]:
s = df['Street Name'].dropna()
s[s.str.contains('110')].value_counts()

Street Name
W 110th St              2970
110th St                2388
E 110th St              2048
WB 110TH AVE/BRINKER     922
110th Ave                704
                        ... 
I/O 110TH STREET           1
S/O 1100 BARBEY ST         1
S/W C/O 110 ST             1
OPP 1101 2ND AVENUE        1
E 110  ST                  1
Name: count, Length: 73, dtype: int64

In [187]:
s[s.str.contains('BWAY') | s.str.contains('BROADWAY')].value_counts()

Street Name
SB BROADWAY @ 252ND     21939
NB BROADWAY @ W 228T    13367
BROADWAY                10771
SB BROADWAY @ W 196T     6623
NB BROADWAY @ W 120T     5691
                        ...  
S/O BROADWAY                1
S/O 6601 BROADWAY           1
R/O 1785 BROADWAY           1
S/O 5825 BROADWAY           1
F/O 5141 BROADWAY           1
Name: count, Length: 181, dtype: int64

Does the `Registration State` column contain inconsistent values?

In [188]:
df['Registration State'].value_counts()

Registration State
NY    9753643
NJ    1096110
PA     338779
FL     174056
CT     165205
       ...   
PE         18
SK          8
MX          7
NT          3
YT          2
Name: count, Length: 68, dtype: int64